In [23]:
import os
import numpy as np
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from sympy.codegen.cfunctions import log10
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch 
from torchvision import transforms
from tqdm import tqdm
from PIL import Image
import time
from sklearn.metrics import accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import cv2
from typing import Dict, List
import csv
from dataclasses import dataclass
import argparse

In [24]:
if torch.cuda.is_available():
    print(f"GPU kullanlabilir. Cihaz: {torch.cuda.get_device_name(0)}")
else:
    print("GPU kullanılabilir değil, CPU kullanılacak.")

GPU kullanlabilir. Cihaz: NVIDIA GeForce GTX 1650 Ti


In [25]:
CONFIG = {
    "root":      r"C:\Users\tuann\Desktop\Sanal Kıyafet Deneme Uygulaması\datasets",
    "train_csv": r"C:\Users\tuann\Desktop\Sanal Kıyafet Deneme Uygulaması\datasets\train_pairs_temiz.csv",

    "epochs":     25,
    "batch_size": 4,
    "img_size":   64,
    "lr":         2e-4,
    "out_dir":    "models/gan_tryon",
    "log_dir":    "logs",
}

In [26]:
def read_rgb(path: str, size: int) -> torch.Tensor:
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 127.5 - 1.0
    img = np.transpose(img, (2, 0, 1))
    return torch.from_numpy(img)


def read_mask(path: str, size: int) -> torch.Tensor:
    mask = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(path)
    mask = cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, axis=0)
    return torch.from_numpy(mask)

In [27]:
class PairTryOnDataset(Dataset):
    REQUIRED = {
        "person_image", "cloth_image", "cloth_mask",
        "pose_map", "agnostic_image", "target_image",
    }

    def __init__(self, csv_path: str, root: str, size: int):
        self.root = root
        self.size = size
        self.rows: List[Dict[str, str]] = []

        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.rows.append(row)

        if not self.rows:
            raise ValueError("CSV bos!")

        missing = self.REQUIRED - set(self.rows[0].keys())
        if missing:
            raise ValueError(f"CSV eksik kolonlar: {sorted(missing)}")

    def __len__(self):
        return len(self.rows)

    def _abs(self, rel: str) -> str:
        return os.path.join(self.root, rel.replace("/", os.sep))

    def __getitem__(self, idx):
        row = self.rows[idx]
        return {
            "person":     read_rgb (self._abs(row["person_image"]),   self.size),
            "cloth":      read_rgb (self._abs(row["cloth_image"]),    self.size),
            "cloth_mask": read_mask(self._abs(row["cloth_mask"]),     self.size),
            "pose":       read_rgb (self._abs(row["pose_map"]),       self.size),
            "agnostic":   read_rgb (self._abs(row["agnostic_image"]), self.size),
            "target":     read_rgb (self._abs(row["target_image"]),   self.size),
        }

In [28]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x: torch.Tensor):
        feat = self.conv(x)
        return feat, self.pool(feat)


class Up(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_ch * 2, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = torch.nn.functional.interpolate(
                x, size=skip.shape[-2:], mode="bilinear", align_corners=False
            )
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

In [29]:
class TryOnGenerator(nn.Module):
    """
    Giris kanallari:
    - agnostic: 3
    - cloth: 3
    - cloth_mask: 1
    - pose: 3
    Toplam: 10 kanal
    """

    def __init__(self, in_ch: int = 10, base: int = 64):
        super().__init__()
        self.d1 = Down(in_ch, base)
        self.d2 = Down(base, base * 2)
        self.d3 = Down(base * 2, base * 4)
        self.mid = ConvBlock(base * 4, base * 8)
        self.u3 = Up(base * 8, base * 4)
        self.u2 = Up(base * 4, base * 2)
        self.u1 = Up(base * 2, base)
        self.out = nn.Conv2d(base, 3, kernel_size=1)
        self.tanh = nn.Tanh()

    def forward(
        self,
        agnostic: torch.Tensor,
        cloth: torch.Tensor,
        cloth_mask: torch.Tensor,
        pose: torch.Tensor,
    ) -> torch.Tensor:
        x = torch.cat([agnostic, cloth, cloth_mask, pose], dim=1)
        s1, x = self.d1(x)
        s2, x = self.d2(x)
        s3, x = self.d3(x)
        x = self.mid(x)
        x = self.u3(x, s3)
        x = self.u2(x, s2)
        x = self.u1(x, s1)
        x = self.out(x)
        return self.tanh(x)

In [30]:
class PatchDiscriminator(nn.Module):
    def __init__(self, in_ch: int = 13, base: int = 64):
        """
        Discriminator girisi:
        - condition: agnostic(3) + cloth(3) + cloth_mask(1) + pose(3) = 10
        - image: target veya fake = 3
        Toplam: 13
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base, base * 2, 4, 2, 1),
            nn.BatchNorm2d(base * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base * 2, base * 4, 4, 2, 1),
            nn.BatchNorm2d(base * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base * 4, 1, 4, 1, 1),
        )

    def forward(
        self,
        agnostic: torch.Tensor,
        cloth: torch.Tensor,
        cloth_mask: torch.Tensor,
        pose: torch.Tensor,
        image: torch.Tensor,
    ) -> torch.Tensor:
        x = torch.cat([agnostic, cloth, cloth_mask, pose, image], dim=1)
        return self.net(x)


def sobel_edge_map(img: torch.Tensor) -> torch.Tensor:
    gray = 0.299 * img[:, 0:1] + 0.587 * img[:, 1:2] + 0.114 * img[:, 2:3]
    kx = torch.tensor(
        [[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=img.dtype, device=img.device
    ).view(1, 1, 3, 3)
    ky = torch.tensor(
        [[1, 2, 1], [0, 0, 0], [-1, -2, -1]], dtype=img.dtype, device=img.device
    ).view(1, 1, 3, 3)
    ex = torch.nn.functional.conv2d(gray, kx, padding=1)
    ey = torch.nn.functional.conv2d(gray, ky, padding=1)
    return torch.sqrt(ex * ex + ey * ey + 1e-6)

In [31]:
graphics_pth = r"graphics"
if not os.path.exists(graphics_pth):
    os.makedirs(graphics_pth)
graphic_files = [f for f in os.listdir(graphics_pth) if f.startswith("graph") and f.endswith("png")]
if not graphic_files:
    graphic_filename = "graph_1.png"
else:
    graphic_max_suffix = max([int(f.split('_')[-1].split('.')[0]) for f in graphic_files])
    graphic_filename = f"graph_{graphic_max_suffix + 1}.png"
graphic_save_path = os.path.join(graphics_pth, graphic_filename)

log_pth = r"logs"
log_files = [f for f in os.listdir(log_pth) if f.startswith("log") and f.endswith("csv")]
if not log_files:
    log_filename = "log_1.csv"
else:
    log_max_suffix = max([int(f.split('_')[-1].split('.')[0]) for f in log_files])
    log_filename = f"log_{log_max_suffix + 1}.csv"
log_save_path = os.path.join(log_pth, log_filename)

In [32]:
@dataclass
class LossWeights:
    adv: float = 0.5
    l1: float = 15.0
    edge: float = 2.0


def get_next_filename(folder, prefix, ext):
    os.makedirs(folder, exist_ok=True)
    files = [f for f in os.listdir(folder) if f.startswith(prefix + "_") and f.endswith("." + ext)]
    if not files:
        return os.path.join(folder, f"{prefix}_1.{ext}")
    max_n = max(int(f.replace(prefix + "_", "").replace("." + ext, "")) for f in files)
    return os.path.join(folder, f"{prefix}_{max_n + 1}.{ext}")


def save_loss_graph(log_path, graph_path):
    import matplotlib.pyplot as plt
    df = pd.read_csv(log_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(df["epoch"], df["g_total"], label="G Total")
    ax1.plot(df["epoch"], df["g_adv"],   label="G Adv")
    ax1.plot(df["epoch"], df["g_l1"],    label="G L1")
    ax1.plot(df["epoch"], df["g_edge"],  label="G Edge")
    ax1.set_title("Generator Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(df["epoch"], df["d_total"], label="D Total", color="red")
    ax2.set_title("Discriminator Loss")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(graph_path, dpi=150)
    plt.close()
    print(f"Grafik kaydedildi: {graph_path}")


def save_sample(batch, fake, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    target  = batch["target"][0].detach().cpu().numpy()
    fake_np = fake[0].detach().cpu().numpy()
    target  = ((np.transpose(target,  (1, 2, 0)) + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    fake_np = ((np.transpose(fake_np, (1, 2, 0)) + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    vis = np.concatenate([target, fake_np], axis=1)
    cv2.imwrite(out_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))


def train_one_epoch(loader, g, d, g_opt, d_opt, bce, l1, weights, device):
    g.train(); d.train()
    meter = {"g_total": 0., "g_adv": 0., "g_l1": 0., "g_edge": 0., "d_total": 0.}
    n = 0

    for batch in loader:
        agnostic   = batch["agnostic"].to(device)
        cloth      = batch["cloth"].to(device)
        cloth_mask = batch["cloth_mask"].to(device)
        pose       = batch["pose"].to(device)
        target     = batch["target"].to(device)

        fake = g(agnostic, cloth, cloth_mask, pose)

        # Discriminator
        d_opt.zero_grad()
        pred_real = d(agnostic, cloth, cloth_mask, pose, target)
        pred_fake = d(agnostic, cloth, cloth_mask, pose, fake.detach())
        d_loss = 0.5 * (
            bce(pred_real, torch.ones_like(pred_real)) +
            bce(pred_fake, torch.zeros_like(pred_fake))
        )
        d_loss.backward()
        d_opt.step()

        # Generator
        g_opt.zero_grad()
        pred_fake_g = d(agnostic, cloth, cloth_mask, pose, fake)
        g_adv  = bce(pred_fake_g, torch.ones_like(pred_fake_g))
        g_l1   = l1(fake, target)
        g_edge = l1(sobel_edge_map(fake), sobel_edge_map(target))
        g_total = weights.adv * g_adv + weights.l1 * g_l1 + weights.edge * g_edge
        g_total.backward()
        g_opt.step()

        meter["g_total"] += g_total.item()
        meter["g_adv"]   += g_adv.item()
        meter["g_l1"]    += g_l1.item()
        meter["g_edge"]  += g_edge.item()
        meter["d_total"] += d_loss.item()
        n += 1

    return {k: v / max(n, 1) for k, v in meter.items()}


def train(config: dict = CONFIG):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Cihaz: {device}")

    model_dir    = "models"
    graphics_dir = "graphics"
    log_dir      = config.get("log_dir", "logs")

    os.makedirs(model_dir,    exist_ok=True)
    os.makedirs(graphics_dir, exist_ok=True)
    os.makedirs(log_dir,      exist_ok=True)

    log_path   = get_next_filename(log_dir,      "log",           "csv")
    graph_path = get_next_filename(graphics_dir, "graph",         "png")
    g_path     = get_next_filename(model_dir,    "generator",     "pth")
    d_path     = get_next_filename(model_dir,    "discriminator", "pth")

    print(f"Log    : {log_path}")
    print(f"Grafik : {graph_path}")
    print(f"G model: {g_path}")
    print(f"D model: {d_path}")

    dataset = PairTryOnDataset(config["train_csv"], config["root"], config["img_size"])
    loader  = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=False,
    )
    print(f"Dataset: {len(dataset)} ornek, {len(loader)} batch")

    g     = TryOnGenerator().to(device)
    d     = PatchDiscriminator().to(device)
    g_opt = optim.Adam(g.parameters(), lr=config["lr"], betas=(0.5, 0.999))
    d_opt = optim.Adam(d.parameters(), lr=config["lr"], betas=(0.5, 0.999))
    bce   = nn.BCEWithLogitsLoss()
    l1    = nn.L1Loss()
    w     = LossWeights()

    with open(log_path, "w", encoding="utf-8") as f:
        f.write("epoch,g_total,g_adv,g_l1,g_edge,d_total\n")

    best_l1    = float("inf")
    best_epoch = -1

    for epoch in range(1, config["epochs"] + 1):
        m = train_one_epoch(loader, g, d, g_opt, d_opt, bce, l1, w, device)
        print(
            f"[{epoch:>3}/{config['epochs']}] "
            f"G={m['g_total']:.4f}  "
            f"(adv={m['g_adv']:.4f}, l1={m['g_l1']:.4f}, edge={m['g_edge']:.4f})  "
            f"D={m['d_total']:.4f}"
        )

        with open(log_path, "a", encoding="utf-8") as f:
            f.write(
                f"{epoch},{m['g_total']:.6f},{m['g_adv']:.6f},"
                f"{m['g_l1']:.6f},{m['g_edge']:.6f},{m['d_total']:.6f}\n"
            )

        # En iyi modeli kaydet
        if m["g_l1"] < best_l1:
            best_l1    = m["g_l1"]
            best_epoch = epoch
            torch.save(g.state_dict(), g_path)
            torch.save(d.state_dict(), d_path)
            print(f"  --> En iyi model kaydedildi (epoch {epoch}, l1={best_l1:.4f})")

        # Her 5 epoch'ta örnek görsel
        if epoch % 5 == 0:
            sample_batch = next(iter(loader))
            with torch.no_grad():
                fake = g(
                    sample_batch["agnostic"].to(device),
                    sample_batch["cloth"].to(device),
                    sample_batch["cloth_mask"].to(device),
                    sample_batch["pose"].to(device),
                )
            save_sample(
                sample_batch, fake,
                os.path.join(graphics_dir, f"sample_epoch_{epoch}.jpg"),
            )

    print(f"\nEn iyi epoch: {best_epoch}, L1: {best_l1:.4f}")
    save_loss_graph(log_path, graph_path)
    print("Egitim tamamlandi.")


train(CONFIG)

Cihaz: cuda
Log    : logs\log_26.csv
Grafik : graphics\graph_20.png
G model: models\generator_1.pth
D model: models\discriminator_1.pth
Dataset: 11647 ornek, 2912 batch
[  1/25] G=3.1210  (adv=0.8760, l1=0.1267, edge=0.3911)  D=0.6400
  --> En iyi model kaydedildi (epoch 1, l1=0.1267)
[  2/25] G=2.6540  (adv=0.8318, l1=0.1039, edge=0.3397)  D=0.6565
  --> En iyi model kaydedildi (epoch 2, l1=0.1039)
[  3/25] G=2.5183  (adv=0.8271, l1=0.0974, edge=0.3222)  D=0.6588
  --> En iyi model kaydedildi (epoch 3, l1=0.0974)
[  4/25] G=2.4293  (adv=0.8241, l1=0.0931, edge=0.3105)  D=0.6607
  --> En iyi model kaydedildi (epoch 4, l1=0.0931)
[  5/25] G=2.3619  (adv=0.8267, l1=0.0897, edge=0.3014)  D=0.6604
  --> En iyi model kaydedildi (epoch 5, l1=0.0897)
[  6/25] G=2.3174  (adv=0.8301, l1=0.0874, edge=0.2953)  D=0.6604
  --> En iyi model kaydedildi (epoch 6, l1=0.0874)
[  7/25] G=2.2761  (adv=0.8371, l1=0.0852, edge=0.2897)  D=0.6595
  --> En iyi model kaydedildi (epoch 7, l1=0.0852)
[  8/25] G=2